# Midterm Project
- AIGC 5500 0NA
- Advanced Deep Learning
- Summer 2026


### Team Info
HawkAI Detection | Group 1
| Name | Github |
| --- | --- |
| Kevin Joseff Cabrera (Kevin) | https://github.com/kjcabPL |
| Mohd Mujtaba Saighani (Mujtaba) | https://github.com/saigha1 |
| Thiago Segantini Nogeuira (Thiago) | https://github.com/thiaseg |
| Sayamon Sittiprom (Saya) | https://github.com/sittiprom |
| Yao-Fu Yang (Andy) | https://github.com/yaofu-yang |

### 1. Overview (Project Instructions)
In the activation functions and gradient descent module, you studied variations of gradient descent
and the hyperparameters that govern training dynamics. This project puts that knowledge to work:
you will systematically compare three modern optimizers on a real image-classification task, analyze
how each one behaves under different hyperparameter settings, and communicate your findings in a
live group presentation.
- Objective: Implement, tune, and compare Adam, RMSprop, and AdamW on a feedforward
neural network trained on the KMNIST dataset, and draw evidence-based conclusions about
their relative strengths and trade-offs.
- Importance: Choosing the wrong optimizer or leaving hyperparameters at default values is
one of the most common causes of poor model performance. This project builds the analytical
habits you will carry into every future deep learning task.

### 2. Dataset (Project Instructions)
Use the KMNIST (Kuzushiji-MNIST) dataset, available directly through PyTorch:
| Property | Details |
|---|---|
|Training Samples| 60,000 Images|
|Test Samples| 10,000 Images|
|Image Size | 28 x 28 Pixels, Greyscale|
|Classes| 10 (Japanese Hiragana Characters)|
|Complexity| Higher than MNIST; class boundaries overlap|


In [17]:
# Handle all imports first
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# Kuzushiji-MNIST Dataset
from torchvision.datasets import KMNIST
from torch.utils.data import DataLoader, Subset
import numpy as np

In [2]:
# Define data preparation constants
MANUAL_SEED = 498
ROOT ='./kuzushiji_data'

In [3]:
# Load the data and complete basic exploration

# Setting the random seed
torch.manual_seed(MANUAL_SEED)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Training data as a single batch to compute mean and standard deviation
# Define a transformation to Tensor
transform = transforms.Compose([transforms.ToTensor()])
trainset = KMNIST(root=ROOT, train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=len(trainset), shuffle=True)
data = next(iter(trainloader))
mean = data[0].mean().item()
stddev = data[0].std().item()
print(f"The mean is : {mean}")
print(f"The standard deviation is: {stddev}")


Using device: cpu
The mean is : 0.19176216423511505
The standard deviation is: 0.3483428359031677


In [4]:
# Check data shape
# (Batch Size, Channel Counts, Row Pixels, Column Pixels)
data[0].size()

torch.Size([60000, 1, 28, 28])

In [9]:
# Prepare the data

# Apply Input Standardization to reduce risk of saturating neurons
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(mean, stddev)])

# Create training and testing sets while applying standardization
trainset = KMNIST(root=ROOT, train=True, download=True, transform=transform)
testset = KMNIST(root=ROOT, train=False, download=True, transform=transform)


# Verify final dimension of all three data sets
print(f"Train set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")

Train set size: 60000
Test set size: 10000


# 3. Model Architecture (Project Instructions)
Implement the following feedforward fully-connected architecture. Do not modify it; the architecture is fixed so that all differences between runs are attributable to the optimizer and its hyperparameters.

|Layer|Configuration|Notes
|---|---|---|
|Input|784 Neurons|Flatten 28x28 image|
|Hidden 1|128 Neurons, ReLU Activation||
|Hidden 1|64 Neurons, ReLU Activation||
|Output|10 Neurons, Softmax activation|One neuron per class|
|Loss function|Cross-Entropy Loss|Appropriate for multi-class classification|


In [ ]:
# Define architectural constants
INPUT_DIM = 784  # Cannot change
HIDDEN_DIM1 = 128  # Cannot change
HIDDEN_DIM2 = 64  # Cannot change
OUTPUT_DIM = 10  # Cannot change

In [ ]:
# Define the forward pass. Final activation is omitted since softmax is part of
# cross-entropy loss function in PyTorch.
class FeedForwardNet(nn.Module):
    def __init__(
        self,
        input_dim:  int = INPUT_DIM,
        hidden_dim1: int = HIDDEN_DIM1,
        hidden_dim2: int = HIDDEN_DIM2,
        output_dim: int = OUTPUT_DIM
    ) -> None:
        super().__init__()
        # First Hidden Layer
        self.hidden1 = nn.Linear(input_dim, hidden_dim1)
        # Apply BatchNorm to first hidden layer
        self.batchNorm1 = nn.BatchNorm1d(num_features=hidden_dim1)
        # Apply random deactivation of neurons to reduce overfitting
        self.dropout1 = nn.Dropout(p=0.2)

        # Second Hidden Layer
        self.hidden2 = nn.Linear(hidden_dim1, hidden_dim2)
        # Apply BatchNorm to second hidden layer
        self.batchNorm2 = nn.BatchNorm1d(num_features=hidden_dim2)
        # Apply random deactivation of neurons to reduce overfitting
        self.dropout2 = nn.Dropout(p=0.2)

        # Final Output Layer
        self.output = nn.Linear(hidden_dim2, output_dim)

        # Customized function to initialize weights for each layer.
        self._initialize_weights()

    # Applies Weight Initialization technique.
    def _initialize_weights(self) -> None:
        # Kaiming (He) initialization for first hidden layer
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity='relu')
        nn.init.constant_(self.hidden1.bias, 0.0)

        # Same for second hidden layer
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity='relu')
        nn.init.constant_(self.hidden2.bias, 0.0)

        # Xavier (Glorot) initialization for the output layer (no activation).
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.constant_(self.output.bias, 0.0)

    # Completes the forward pass
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Flatten the image from 28x28 to 784 pixels
        x = torch.flatten(x, start_dim=1)

        # First layers (hidden, batchnorm, relu activation, dropout)
        x = self.hidden1(x)
        x = self.batchNorm1(x)
        x = F.relu(x)
        x = self.dropout1(x)

        # Second Layers (hidden, batchnorm, relu activation, dropout)
        x = self.hidden2(x)
        x = self.batchNorm2(x)
        x = F.relu(x)
        x = self.dropout2(x)

        return self.output(x)    # Raw logits; softmax applied by loss function.

# Initializes the model.
model = FeedForwardNet()

# 4. Optimizers Under Investigation (Project Instructions)
You will train and evaluate the same fixed architecture using each of the three optimizers below. For 
every optimizer, perform a structured hyperparameter search (see Section 5) and report results for 
both the default configuration and the best configuration found. 
 
### 4.1  Adam (Adaptive Moment Estimation) 
- Key idea: Maintains separate adaptive learning rates for each parameter by combining first-
moment (momentum) and second-moment (uncentred variance) estimates. 
- Default settings: lr = 0.001, β1 = 0.9, β2 = 0.999, ε = 1e-8. 
- Investigate: The effect of learning rate and β1 on convergence speed and final accuracy. 
 
### 4.2  RMSprop (Root Mean Square Propagation) 
- Key idea: Divides the learning rate by a running average of the squared gradient magnitudes, 
preventing updates from growing too large. 
- Default settings: lr = 0.01, α = 0.99, ε = 1e-8. 
- Investigate: The effect of the decay factor α and learning rate on training stability. 
 
### 4.3  AdamW (Adam with Decoupled Weight Decay) 
- Key  idea:  Corrects a known flaw in Adam’s L2 regularization by decoupling weight decay 
from the adaptive learning-rate update, leading to better generalization. 
- Default settings: lr = 0.001, β1 = 0.9, β2 = 0.999, ε = 1e-8, weight_decay = 0.01. 
- Investigate: The effect of weight_decay and learning rate on the gap between training and 
test accuracy. 

### Branch TODO 1: Research the Assigned Optimizer
The project instructions give a brief overview of the key ideas, default settings, and investigation pathways for each optimizer. Start by completing the investigation tasks but go deeper to find other intereting and unique parameters/features of each optimizer (including ones not covered in lecture).

Action Item: Add a document in the branch with the findings, or include the findings directly in the notebook. For reference, the final PDF report requires the following: "Optimizer descriptions and theoretical background"

In [ ]:
# Define adjustable constants
EPOCHS = 20  # Min 20
BATCH_SIZE = 64  # May change
K_FOLD = 5

# Methodology (Project Instructions)
### 5.1  Hyperparameter Search 
For each optimizer, evaluate at least four (4) distinct hyperparameter configurations. You may use 
grid search, random search, or a manual systematic approach. Document your search strategy and 
justify your final selection. 
 
At minimum, vary the following for each optimizer: 
- Learning  rate: try values  spanning  at  least  two  orders  of magnitude (e.g.,  0.1,  0.01,  0.001, 
0.0001). 
- At least one optimizer-specific hyperparameter (e.g., β1 for Adam/AdamW, decay factor for 
RMSprop).

### 5.2  Cross-Validation 
- Apply 5-fold cross-validation on the training set to estimate generalization performance. 
- Report  the  mean  and  standard  deviation  of  validation  accuracy  across  folds  for  each 
configuration. 
- Use the best configuration identified through cross-validation to train a final model on the full 
training set.

### 5.3  Training Protocol 
- Train  for  a  fixed  number  of  epochs  (minimum  20)  across  all  configurations  to  ensure  fair 
comparison. 
- Use the same random seed for all runs to control for initialization variance. 
- Record training loss, validation loss, and validation accuracy at each epoch. 
- Evaluate the final model on the held-out test set; report test accuracy and test loss.

### 5.4  Metrics to Record
|Metric|Description|
|---|---|
|Training Accuracy|Accuracy on the training set at the final epoch|
|Validation Accuracy|Mean cross-validation accuracy (and std dev) across 5 folds|
|Test Accuracy|Accuracy on the held-out test set |
|Final Loss|Training and test loss at the end of training|
|Convergence Speed|Number of epochs to reach a target validation accuracy (e.g., 
80%)|
|Training Time|Wall-clock time for a full training run|

In [ ]:
# Define a reusable run Epoch function for both training & evaluation loops
def run_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    device:    torch.device,
    training:  bool
) -> tuple[float, float]:
    """Run one full pass over the DataLoader.
    Pass optimizer=None during evaluation.
    Returns (mean loss, accuracy).
    """
    model.train() if training else model.eval()

    total_loss    = 0.0
    total_correct = 0
    total_samples = 0

    # No gradient/back propagation if in evaluation mode.
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            # Cross-entropy loss does not need one-hot targets in PyTorch.
            loss = criterion(outputs, targets)

            # Only run for training mode. Evaluation mode skips this.
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Accumulate metrics.
            _, predicted   = torch.max(outputs, dim=1)
            total_correct += (predicted == targets).sum().item()
            total_samples += targets.size(0)
            total_loss    += loss.item()

    return total_loss / len(loader), total_correct / total_samples